# 6 · Execution and impact

What happens between the decision and the fill.

A historical print tells you the price, never what your order would have
done to it. Here the same seed runs twice, once with your orders and once
without, so every fill is priced against the market where you never traded.

That counterfactual is what arrival price, VWAP and fitted impact models
approximate. Here it is measured directly.

In [1]:
import pretium as pt

universe = pt.Universe.random(20, seed=111)

execution = pt.tca.analyse(pt.baselines.Momentum(),
                           seed=7, universe=universe, days=3)

print("fills           :", len(execution.fills))
print("partial fills   :", len(execution.partial_fills()))
print("shortfall (bps) :", round(execution.shortfall_bps(), 3))

fills           : 156
partial fills   : 0
shortfall (bps) : 7.225


## The counterfactual run

`actual_final` is the closing price of each instrument. `baseline_final` is
the same market with the agent's orders removed: same seed, same draws,
everything else identical. Where they differ, that difference is the
agent's footprint.

In [2]:
tickers = [i.ticker for i in universe]
actual, baseline = execution.actual_final, execution.baseline_final

print(f"{'ticker':8s} {'with orders':>13s} {'without':>13s} {'difference':>12s}")
for t, a, b in zip(tickers, actual, baseline):
    if a != b:
        print(f"{t:8s} {a:13.4f} {b:13.4f} {a - b:12.4f}")

untouched = sum(1 for a, b in zip(actual, baseline) if a == b)
print(f"\n{untouched} of {len(tickers)} names were not moved at all.")

ticker     with orders       without   difference
AAA           194.6500      195.5000      -0.8500
AAB            11.7100       11.6900       0.0200
AAD             6.3400        6.3300       0.0100
AAG           278.9700      279.0600      -0.0900
AAJ           126.8600      126.8500       0.0100
AAM           101.8100      101.8300      -0.0200
AAO            10.3900       10.4100      -0.0200
AAP           138.4000      145.4800      -7.0800
AAR             4.6700        4.6200       0.0500
AAT           495.3400      495.2200       0.1200

10 of 20 names were not moved at all.


## Which names moved

`moved()` reports every instrument whose final price differs between the two
runs, and it reports that difference in basis points, not in currency. The
column printed below as `price moved` is therefore a bps figure, which is why
it agrees with `impact_bps` to the rounding: both are the same measurement.
Bps is what lets a displacement on AAR, trading below 5, sit in the same
column as one on AAT, trading near 495. For the move in currency, read the
table above.

In [3]:
moved = execution.moved()
worst = sorted(moved.items(), key=lambda kv: -abs(kv[1]))[:6]

print(f"{'ticker':8s} {'price moved':>13s} {'impact (bps)':>14s}")
for ticker, delta in worst:
    print(f"{ticker:8s} {delta:13.4f} {execution.impact_bps(ticker):14.3f}")

ticker     price moved   impact (bps)
AAP          -486.6648       -486.665
AAR           108.2251        108.225
AAA           -43.4783        -43.478
AAO           -19.2123        -19.212
AAB            17.1086         17.109
AAD            15.7978         15.798


## Where the cost fell

`by_step` attributes the shortfall across decision points and `by_ticker`
across names. Positive is cost paid. A negative step, like step 7 below, is
one where the fills came in better than the untraded market: trading back
into your own impact gives some of it back. Cost concentrated in a few steps
says something about the schedule.

In [4]:
steps = execution.by_step()
print("by step:")
for step, value in steps:
    bar = "#" * int(min(40, abs(value) / max(1, max(abs(v) for _, v in steps)) * 40))
    print(f"  step {step:3d} {value:12,.0f}  {bar}")

by step:
  step   6        1,015  #################################
  step   7         -317  ##########
  step   8          512  ################
  step   9          146  ####
  step  10            5  
  step  11          323  ##########
  step  12          786  #########################
  step  13          135  ####
  step  14          681  ######################
  step  15          811  ##########################
  step  16          935  ##############################
  step  17        1,220  ########################################


In [5]:
by_ticker = execution.by_ticker()
top = sorted(by_ticker.items(), key=lambda kv: -abs(kv[1]))[:6]
print("largest contributions by name:")
for ticker, value in top:
    print(f"  {ticker:8s} {value:12,.0f}")

largest contributions by name:
  AAO             1,154
  AAN               713
  AAH               658
  AAS               593
  AAP               500
  AAR               449


## Partial fills

An order asking for more than the book holds at a price does not silently
receive it. Queue position and depth decide what you actually get.

In [6]:
if execution.partial_fills():
    print(f"{len(execution.partial_fills())} partial fills")
    for f in execution.partial_fills()[:5]:
        print("  ", f)
else:
    print("No partial fills at this size; the book absorbed every order.")
    print("Raising participation or size is how you find the edge of that;")
    print("the whole point is that the edge exists and is measurable.")

No partial fills at this size; the book absorbed every order.
Raising participation or size is how you find the edge of that;
the whole point is that the edge exists and is measurable.


## Comparing three algorithms

Same market, same seed, with the counterfactual computed for each.

In [7]:
candidates = {
    "momentum":       pt.baselines.Momentum(),
    "mean reversion": pt.baselines.MeanReversion(),
    "buy and hold":   pt.baselines.BuyAndHold(),
}

print(f"{'algorithm':16s} {'shortfall bps':>14s} {'fills':>7s} {'partials':>9s}")
for name, agent in candidates.items():
    ex = pt.tca.analyse(agent, seed=7, universe=universe, days=3)
    print(f"{name:16s} {ex.shortfall_bps():14.3f} {len(ex.fills):7d} "
          f"{len(ex.partial_fills()):9d}")

algorithm         shortfall bps   fills  partials
momentum                  7.225     156         0
mean reversion           -1.744     158         0
buy and hold             13.191      20         0


## Provenance

An execution result carries the seed and model fingerprint, so a TCA number
can be cited.

In [8]:
print("seed             :", execution.seed)
print("model fingerprint:", execution.model_fingerprint)

seed             : 7
model fingerprint: pt-v12


## Caveats

**Volume changes were this notebook's structural caveat, and are not any
more.** A ceiling in the engine capped a name's volume response at a four
percent daily move, so a violent day traded like a quiet one. Be precise
about the horizon that cost, because it was not a one-year failure:
`volume_change_acf1` was already inside its 252-day band on `pt-v10`.
Notebook 04 scores that preset at 13 of 14 at the certified horizon and this
is not the row it gives up. What it missed was the tighter band re-derived
at 504 days, which is why the gap had been rewritten around the horizon
rather than dropped. `pt-v12`, the preset fingerprinted above, raised the
ceiling from four percent to twelve; the statistic came inside at BOTH
horizons and the volume-change gap was retired from the realism envelope.
Schedule-shape conclusions no longer carry that warning at either horizon,
and depth, queue and impact conclusions are sound as they were before.
Notebook 04 prints the panel this rests on. What the envelope still forbids
lives in five other gaps -- horizon, decay-shape, scenario-magnitude,
macro-range, roster-concentration -- none of them about the fill mechanics
measured here.

**Single venue, no latency.** One book per name, orders arrive instantly,
and no strategic counterparties adapt to you.

Full documentation: <https://simoncoombes.github.io/pretium/>